In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)

        records.extend(page["data"])

        if page["next"]:
            url = urljoin(url, page["next"])
        else:
            url = None

    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(orders)

1852

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

save_json(orders, os.path.join(RAW_DIR, "orders.json"))


In [0]:
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(path, "wb") as f:
        f.write(response.content)

    return len(response.content)

In [0]:
download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,
'teams.csv'))

801

In [0]:
#5.

def extract(raw_dir: str) -> dict:
    #1 porudzbine preko API-ja
    orders = fetch_all(BASE_URL + "orders/page-1.json")
    save_json(orders, os.path.join(raw_dir, "orders.json"))

    orders_df = pd.json_normalize(orders, sep="_")
    orders_df = orders_df.rename(
        columns={
            "customer_name": "customer",
            "customer_city": "city"
        }
    )

    #2 fajlovi
    download_file(
        FILES_URL + "products.csv",
        os.path.join(raw_dir, "products.csv")
    )

    download_file(
        FILES_URL + "teams.csv",
        os.path.join(raw_dir, "teams.csv")
    )

    download_file(
        FILES_URL + "matches.parquet",
        os.path.join(raw_dir, "matches.parquet")
    )

    #3 porudzbine u DateFrame
    products_df = pd.read_csv(
        os.path.join(raw_dir, "products.csv")
    )

    teams_df = pd.read_csv(
        os.path.join(raw_dir, "teams.csv")
    )

    matches_df = pd.read_parquet(
        os.path.join(raw_dir, "matches.parquet")
    )

    #4 vracanje recnika
    return {
        "orders": orders_df,
        "products": products_df,
        "teams": teams_df,
        "matches": matches_df
    }

data = extract(RAW_DIR)

print(data["orders"].shape)
print(data["products"].shape)
print(data["teams"].shape)
print(data["matches"].shape)

(1852, 8)
(108, 7)
(20, 5)
(120, 9)


In [0]:
#6.

def clean_products(df: pd.DataFrame) -> pd.DataFrame:

    # uklanjanje duplikata 
    df = df.drop_duplicates(subset="product_id", keep="first").copy()

    # uklanja razmake, pretvaranja u velika slova
    df["team_code"] = (
        df["team_code"]
        .fillna("")
        .str.strip()
        .str.upper()
    )

    # prazan string vratiti na NA
    df["team_code"] = df["team_code"].replace("", pd.NA)

    # uklanjanje razmaka i mala slova
    df["category"] = (
        df["category"]
        .str.strip()
        .str.lower()
    )

    # price
    df["price"] = (
        df["price"]
        .astype(str)
        .str.replace(" RSD", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # active
    df["active"] = (
        df["active"]
        .astype(str)
        .str.lower()
        .eq("yes")
    )

    return df

clean_df = clean_products(data["products"])

print(clean_df.shape[0])          # 105
print(clean_df["price"].dtype)    # float64
print(clean_df["active"].sum())   # 101

print(
    clean_df.loc[
        clean_df["product_id"] == 1003,
        ["team_code", "category", "price", "active"]
    ]
)

105
float64
101
  team_code category   price  active
2       NPM  scarves  1490.0    True


In [0]:
#7.

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df["order_id"] = df["order_id"].str.strip().str.upper()
    df["customer"] = df["customer"].str.strip().str.title()
    df["city"] = df["city"].fillna("Unknown").str.strip().str.upper()
    df["payment"] = df["payment"].str.strip().str.upper()

    return df

orders_clean = clean_orders(data["orders"])

print(orders_clean.head())

    order_id  order_date  product_id quantity unit_price payment  \
0  NSS-00001  2026-08-15        1002        1     6990.0    CASH   
1  NSS-00002  2026-08-15        1094        3     1290.0    CARD   
2  NSS-00003  2026-08-15        1041        1     6990.0    CARD   
3  NSS-00004  2026-08-15        1075        1      990.0    CASH   
4  NSS-00005  2026-08-15        1044        1     1290.0    CARD   

            customer       city  
0  Nikola Stojanović  ZRENJANIN  
1    Jovana Pavlović      ČAČAK  
2      Sara Živković    BEOGRAD  
3      Sara Đorđević   SUBOTICA  
4     Stefan Popović        NIŠ  


In [0]:
#8

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:

    unique = df.drop_duplicates(
        subset="order_id",
        keep="first"
    )

    duplicates = df[
        df.duplicated(
            subset="order_id",
            keep="first"
        )
    ]

    return unique, duplicates

unique_orders, duplicate_orders = remove_duplicate_orders(
    data["orders"]
)

print(unique_orders.shape[0])
print(duplicate_orders.shape[0])


1840
12


In [0]:
#9

def validate_order(row: dict, product_ids: set) -> str:

    # order_id
    if not str(row["order_id"]).startswith("NSS-"):
        return "invalid_order"

    # quantity nedostaje
    if pd.isna(row["quantity"]):
        return "missing_quantity"

    # quantity ceo broj > 0
    quantity = pd.to_numeric(row["quantity"], errors="coerce")

    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return "invalid_quantity"

    # unit_price broj > 0
    price = pd.to_numeric(row["unit_price"], errors="coerce")

    if pd.isna(price) or price <= 0:
        return "invalid_price"

    # product_id mora da postoji
    if row["product_id"] not in product_ids:
        return "unknown_product"

    return "OK"

In [0]:
product_ids = {1001, 1002, 1003}
print(validate_order(
    {
        "order_id": "NSS-1",
        "quantity": 2,
        "unit_price": 100,
        "product_id": 1003
    },
    product_ids
))

OK


In [0]:
#10.

def transform_orders(df: pd.DataFrame,products: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # tipovi
    df["quantity"] = df["quantity"].astype(int)
    df["unit_price"] = df["unit_price"].astype(float)
    df["order_date"] = pd.to_datetime(df["order_date"])

    # total
    df["total"] = df["quantity"] * df["unit_price"]

    # kolone iz proizvoda
    products_small = products[
        ["product_id", "product_name", "team_code", "category"]
    ]

    # merge
    df = df.merge(
        products_small,
        on="product_id",
        how="left"
    )

    return df

In [0]:
products_clean = clean_products(data["products"])

product_ids = set(products_clean["product_id"])

unique_orders, duplicate_orders = remove_duplicate_orders(
    data["orders"]
)

orders_df = unique_orders.copy()

orders_df["status"] = orders_df.apply(
    lambda row: validate_order(row, product_ids),
    axis=1
)

valid_orders = orders_df[orders_df["status"] == "OK"].copy()

In [0]:
orders_transformed = transform_orders(
    valid_orders.drop(columns=["status"]),
    products_clean
)

print(orders_transformed.shape)
print(valid_orders.shape)

(1800, 12)
(1800, 9)


In [0]:
#11.

def split_orders(df: pd.DataFrame,
                 products: pd.DataFrame) -> tuple:

    # čišćenje porudžbina
    df = clean_orders(df)

    # uklanjanje duplikata
    unique_orders, duplicate_orders = remove_duplicate_orders(df)

    duplicate_orders = duplicate_orders.copy()
    duplicate_orders["reason"] = "duplicate"

    # validacija
    product_ids = set(products["product_id"])

    unique_orders = unique_orders.copy()
    unique_orders["reason"] = unique_orders.apply(
        lambda row: validate_order(row, product_ids),
        axis=1
    )

    # validni
    valid = unique_orders[
        unique_orders["reason"] == "OK"
    ].copy()

    valid = valid.drop(columns=["reason"])

    valid = transform_orders(valid, products)

    # odbijeni
    rejected = unique_orders[
        unique_orders["reason"] != "OK"
    ].copy()

    rejected = pd.concat(
        [rejected, duplicate_orders],
        ignore_index=True
    )

    return valid, rejected

products_clean = clean_products(data["products"])

valid, rejected = split_orders(
    data["orders"],
    products_clean
)

print(valid.shape[0])
print(rejected.shape[0])

print(
    (rejected["reason"] == "duplicate").sum()
)

1800
52
12


In [0]:
#12.

def team_results(matches: pd.DataFrame,
                 teams: pd.DataFrame) -> pd.DataFrame:

    # samo odigrane utakmice
    played = matches[matches["status"] == "finished"].copy()

    # datum bez vremena
    played["kickoff_ts"] = pd.to_datetime(played["kickoff_ts"])

    # team_id -> team_code
    codes = dict(zip(teams["team_id"], teams["code"]))

    # domaćin
    home = pd.DataFrame({
        "match_id": played["match_id"],
        "match_date": played["kickoff_ts"].dt.normalize(),
        "team_code": played["home_team_id"].map(codes),
        "opponent_code": played["away_team_id"].map(codes),
        "goals_for": played["home_goals"],
        "goals_against": played["away_goals"]
    })

    home["result"] = home.apply(
        lambda x: "W" if x["goals_for"] > x["goals_against"]
        else "D" if x["goals_for"] == x["goals_against"]
        else "L",
        axis=1
    )

    home["points"] = home["result"].map({
        "W": 3,
        "D": 1,
        "L": 0
    })

    # gost
    away = pd.DataFrame({
        "match_id": played["match_id"],
        "match_date": played["kickoff_ts"].dt.normalize(),
        "team_code": played["away_team_id"].map(codes),
        "opponent_code": played["home_team_id"].map(codes),
        "goals_for": played["away_goals"],
        "goals_against": played["home_goals"]
    })

    away["result"] = away.apply(
        lambda x: "W" if x["goals_for"] > x["goals_against"]
        else "D" if x["goals_for"] == x["goals_against"]
        else "L",
        axis=1
    )

    away["points"] = away["result"].map({
        "W": 3,
        "D": 1,
        "L": 0
    })

    results = pd.concat(
        [home, away],
        ignore_index=True
    )

    return results

results = team_results(
    data["matches"],
    data["teams"]
)

print(results.shape)

(238, 8)


In [0]:
#13

def league_table(results: pd.DataFrame) -> pd.DataFrame:

    table = results.groupby("team_code").agg(
        played=("team_code", "size"),
        won=("result", lambda x: (x == "W").sum()),
        drawn=("result", lambda x: (x == "D").sum()),
        lost=("result", lambda x: (x == "L").sum()),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        points=("points", "sum")
    ).reset_index()

    table["goal_diff"] = (
        table["goals_for"] - table["goals_against"]
    )

    table = table.sort_values(
        by=["points", "goal_diff", "goals_for"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    table["position"] = range(1, len(table) + 1)

    table = table[
        [
            "position",
            "team_code",
            "played",
            "won",
            "drawn",
            "lost",
            "goals_for",
            "goals_against",
            "goal_diff",
            "points"
        ]
    ]

    return table

In [0]:
results = team_results(
    data["matches"],
    data["teams"]
)

table = league_table(results)

print(table.shape)

print(table.head())

print(
    table[["team_code", "points"]].head(2)
)

(20, 10)
   position team_code  played  won  drawn  lost  goals_for  goals_against  \
0         1       FWR      12    8      2     2         22              8   
1         2       BMC      12    7      2     3         22             13   
2         3       RVA      12    6      5     1         11              4   
3         4       VDC      12    6      4     2         18             13   
4         5       HGA      11    5      4     2         13              6   

   goal_diff  points  
0         14      26  
1          9      23  
2          7      23  
3          5      22  
4          7      19  
  team_code  points
0       FWR      26
1       BMC      23


In [0]:
#14

def match_windows(results: pd.DataFrame) -> pd.DataFrame:

    windows = []

    for d in range(4):

        temp = results[
            ["team_code", "match_date", "result"]
        ].copy()

        temp["date"] = temp["match_date"] + pd.Timedelta(days=d)

        windows.append(temp)

    windows = pd.concat(
        windows,
        ignore_index=True
    )

    windows = windows.rename(
        columns={"result": "window"}
    )

    windows = windows[
        ["team_code", "date", "window"]
    ]

    return windows

results = team_results(
    data["matches"],
    data["teams"]
)

windows = match_windows(results)

print(windows.shape)

(952, 3)


In [0]:
#15

def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:

    revenue = (
        df.groupby(column)["total"]
        .sum()
        .reset_index()
    )

    revenue = revenue.rename(
        columns={"total": "revenue"}
    )

    revenue = revenue.sort_values(
        "revenue",
        ascending=False
    ).reset_index(drop=True)

    return revenue

revenue_team = revenue_by(
    orders_transformed,
    "team_code"
)

print(revenue_team.head())
print(revenue_team.tail(1))

revenue_category = revenue_by(
    orders_transformed,
    "category"
)

print(revenue_category.head())

  team_code   revenue
0       LSB  720420.0
1       FWR  625860.0
2       GMW  625670.0
3       NPM  609330.0
4       AFV  602940.0
   team_code   revenue
19       WFT  234650.0
      category    revenue
0      jerseys  7094410.0
1         caps   958760.0
2      scarves   917880.0
3  accessories   726970.0
4        balls   347380.0
